# 1. Basic Encoder–Decoder Architecture

In [1]:
import torch
import torch.nn as nn

In [3]:
#Encoder
class Encoder(nn.Module):
  def __init__(self, input_size, embedding_dim, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(input_size, embedding_dim)
        self.rnn=nn.GRU(embedding_dim,hidden_size,batch_first=True)

  def forward(self,x):
    embedded =self.embedding(x)
    outputs,hidden=self.rnn(embedded)
    return outputs,hidden



In [8]:
#Decoder
class Decoder(nn.Module):
  def __init__(self,output_size,embedding_dim,hidden_size):
    super().__init__()

    self.embedding=nn.Embedding(output_size,embedding_dim)
    self.rnn=nn.GRU(embedding_dim,hidden_size,batch_first=True)
    self.fc=nn.Linear(hidden_size,output_size)

  def forward(self,x,hidden):
    embedded=self.embedding(x)
    output,hidden=self.rnn(embedded,hidden)

    prediction = self.fc(output)
    return prediction, hidden

In [9]:
input_size = 100
output_size = 100

embedding_dim = 128
hidden_size = 256

encoder = Encoder(
    input_size,
    embedding_dim,
    hidden_size
)

decoder = Decoder(
    output_size,
    embedding_dim,
    hidden_size
)

In [11]:
source = torch.randint(
    0,
    input_size,
    (2, 10)
)

target = torch.randint(
    0,
    output_size,
    (2, 8)
)

encoder_outputs, hidden = encoder(source)

decoder_output, hidden = decoder(
    target, # Input for the decoder, typically a start token followed by previous predictions
    hidden
)

print("Encoder Output:", encoder_outputs.shape)
print("Hidden State:", hidden.shape)
print("Decoder Output:", decoder_output.shape)

Encoder Output: torch.Size([2, 10, 256])
Hidden State: torch.Size([1, 2, 256])
Decoder Output: torch.Size([2, 8, 100])


# 2. Encoder–Decoder for English → French Translation

In [12]:
import torch
import torch.nn as nn

# -----------------------------
# Vocabulary
# -----------------------------

english_vocab = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "i": 3,
    "am": 4,
    "happy": 5
}

french_vocab = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "je": 3,
    "suis": 4,
    "heureux": 5
}

In [13]:
from prompt_toolkit import key_binding
# Reverse vocabulary
reverse_french_vocab={
    value:key
    for key,value in french_vocab.items()
}

In [14]:
#Encoder
class Encoder(nn.Module):
  def __init__(self, input_size, embedding_dim, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(input_size, embedding_dim)
        self.rnn=nn.GRU(embedding_dim,hidden_size,batch_first=True)

  def forward(self,x):
    embedded =self.embedding(x)
    outputs,hidden=self.rnn(embedded)
    return outputs,hidden



In [26]:
#Decoder
class Decoder(nn.Module):
  def __init__(self,output_size,embedding_dim,hidden_size):
    super().__init__()

    self.embedding=nn.Embedding(output_size,embedding_dim)
    self.rnn=nn.GRU(embedding_dim,hidden_size,batch_first=True)
    self.fc=nn.Linear(hidden_size,output_size)

  def forward(self,x,hidden):
    x = x.unsqueeze(1)
    embedded = self.embedding(x)
    output, hidden = self.rnn(embedded, hidden)
    prediction = self.fc(output.squeeze(1))
    return prediction, hidden

In [27]:
#Model
encoder=Encoder(len(english_vocab),128,256) # Changed input_size for encoder to English vocab
decoder=Decoder(len(french_vocab),128,256) # Changed output_size for decoder to French vocab

In [28]:
# Example sentence

sentence = torch.tensor([
    [
        english_vocab["i"],
        english_vocab["am"],
        english_vocab["happy"]
    ]
])

_, hidden = encoder(sentence) # Unpack the encoder's output to get only the hidden state

print("Encoder hidden state:")
print(hidden.shape)

Encoder hidden state:
torch.Size([1, 1, 256])


In [29]:
# Start decoder

input_token = torch.tensor([
    french_vocab["<SOS>"]
])

for step in range(3):

    prediction, hidden = decoder(
        input_token,
        hidden
    )

    next_token = prediction.argmax(1)

    word = reverse_french_vocab[
        next_token.item()
    ]

    print("Generated:", word)

    if word == "<EOS>":
        break

    input_token = next_token

Generated: <EOS>


# Program 1

In [32]:
sentence="Aaliya Fatma heartbeat"
tokens=sentence.split()

print("Input:",sentence)
print("Output:",tokens)

for token in tokens:
  print(token)

Input: Aaliya Fatma heartbeat
Output: ['Aaliya', 'Fatma', 'heartbeat']
Aaliya
Fatma
heartbeat


# Program 2

In [34]:
sentence = "I love Python"

tokens = sentence.split()

print("Input:", sentence)
print("Output:", tokens)
print("Input tokens:")
for token in tokens:
    print(token)

Input: I love Python
Output: ['I', 'love', 'Python']
Input tokens:
I
love
Python


In [37]:
print("\nEncoder:")
encoded=[]
for token in tokens:
  encoded.append(token.upper())
print(encoded)


print("\nDecoder")
decoded=[]
for token in encoded:
  decoded.append(token.lower())
print(decoded)


Encoder:
['I', 'LOVE', 'PYTHON']

Decoder
['i', 'love', 'python']


# Program 3 — English → Hindi Translation

In [38]:
data = [
    ("hello", "नमस्ते"),
    ("good morning", "सुप्रभात"),
    ("how are you", "आप कैसे हैं"),
    ("i am fine", "मैं ठीक हूँ"),
    ("thank you", "धन्यवाद"),
    ("i love python", "मुझे पायथन पसंद है"),
]

In [40]:
#Create dictionaries:
english_words=set()
hindi_words=set()

for english,hindi in data:
  english_words.update(english.split())
  hindi_words.update(hindi.split())

print(english_words)
print(hindi_words)

{'are', 'i', 'am', 'python', 'fine', 'love', 'how', 'good', 'thank', 'hello', 'morning', 'you'}
{'है', 'मैं', 'नमस्ते', 'कैसे', 'ठीक', 'मुझे', 'पसंद', 'पायथन', 'धन्यवाद', 'हूँ', 'हैं', 'आप', 'सुप्रभात'}


In [44]:
#Create vocabulary:
english_vocab = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "<UNK>": 3
}

for word in english_words:
    english_vocab[word] = len(english_vocab)

hindi_vocab = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "<UNK>": 3
}

for word in hindi_words:
    hindi_vocab[word] = len(hindi_vocab)

print(english_vocab)
print(hindi_vocab)

{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'are': 4, 'i': 5, 'am': 6, 'python': 7, 'fine': 8, 'love': 9, 'how': 10, 'good': 11, 'thank': 12, 'hello': 13, 'morning': 14, 'you': 15}
{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'है': 4, 'मैं': 5, 'नमस्ते': 6, 'कैसे': 7, 'ठीक': 8, 'मुझे': 9, 'पसंद': 10, 'पायथन': 11, 'धन्यवाद': 12, 'हूँ': 13, 'हैं': 14, 'आप': 15, 'सुप्रभात': 16}


# Program 4 — Text → Token IDs

In [45]:
sentence = "i love python"

tokens = sentence.split()

ids = []

for token in tokens:

    if token in english_vocab:
        ids.append(english_vocab[token])
    else:
        ids.append(english_vocab["<UNK>"])

print(tokens)
print(ids)

['i', 'love', 'python']
[5, 9, 7]


# Program 5 — Token IDs → Embedding

In [46]:
vocab = [
    "are",
    "i",
    "am",
    "python",
    "fine",
    "love",
    "how",
    "good",
    "thank",
    "hello",
    "morning",
    "you"
]


In [47]:
 # Create word -> index dictionary
# ============================================================

word_to_index = {
    word: index
    for index, word in enumerate(vocab)
}

print("Word to Index:")
print(word_to_index)


# ============================================================
# 4. Create index -> word dictionary
# ============================================================

index_to_word = {
    index: word
    for word, index in word_to_index.items()
}

print("\nIndex to Word:")
print(index_to_word)


# ============================================================
# 5. Check individual words
# ============================================================

print("\nIndexes:")

print("i      ->", word_to_index["i"])
print("love   ->", word_to_index["love"])
print("python ->", word_to_index["python"])
print("hello  ->", word_to_index["hello"])



Word to Index:
{'are': 0, 'i': 1, 'am': 2, 'python': 3, 'fine': 4, 'love': 5, 'how': 6, 'good': 7, 'thank': 8, 'hello': 9, 'morning': 10, 'you': 11}

Index to Word:
{0: 'are', 1: 'i', 2: 'am', 3: 'python', 4: 'fine', 5: 'love', 6: 'how', 7: 'good', 8: 'thank', 9: 'hello', 10: 'morning', 11: 'you'}

Indexes:
i      -> 1
love   -> 5
python -> 3
hello  -> 9


In [48]:
# 6. Convert a sentence into word indexes
# ============================================================

sentence = "i love python"

words = sentence.lower().split()

tokens = [
    word_to_index[word]
    for word in words
]

print("\nSentence:")
print(sentence)

print("\nWords:")
print(words)

print("\nToken IDs:")
print(tokens)


Sentence:
i love python

Words:
['i', 'love', 'python']

Token IDs:
[1, 5, 3]


In [49]:
# ============================================================
# 7. Convert list into PyTorch tensor
# ============================================================

tokens = torch.tensor(tokens)

print("\nPyTorch Tensor:")
print(tokens)

print("Tensor shape:", tokens.shape)


PyTorch Tensor:
tensor([1, 5, 3])
Tensor shape: torch.Size([3])


In [50]:
# 8. Create Embedding layer
# ============================================================

vocab_size = len(vocab)

embedding_dim = 8

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)


# ============================================================
# 9. Pass token IDs into embedding
# ============================================================

output = embedding(tokens)


# ============================================================
# 10. Display embedding vectors
# ============================================================

print("\nEmbedding Output:")
print(output)


Embedding Output:
tensor([[-0.8764,  0.2208,  1.2359, -0.7366,  1.4662, -0.5942, -1.4383, -1.2237],
        [-0.7029, -2.1166,  0.0182, -0.1414, -0.9872,  0.5822,  1.0233, -0.2544],
        [-0.8552, -1.0608,  0.0716,  1.0493,  1.4282, -0.8662,  0.9909,  0.3830]],
       grad_fn=<EmbeddingBackward0>)


In [51]:
# ============================================================
# 11. Display shape
# ============================================================

print("\nEmbedding Shape:")
print(output.shape)


# ============================================================
# 12. Display each word with its embedding
# ============================================================

print("\nWord + Token ID + Embedding:")

for word, token_id, vector in zip(words, tokens, output):

    print("\nWord:", word)

    print("Token ID:", token_id.item())

    print("Embedding:")
    print(vector)


Embedding Shape:
torch.Size([3, 8])

Word + Token ID + Embedding:

Word: i
Token ID: 1
Embedding:
tensor([-0.8764,  0.2208,  1.2359, -0.7366,  1.4662, -0.5942, -1.4383, -1.2237],
       grad_fn=<UnbindBackward0>)

Word: love
Token ID: 5
Embedding:
tensor([-0.7029, -2.1166,  0.0182, -0.1414, -0.9872,  0.5822,  1.0233, -0.2544],
       grad_fn=<UnbindBackward0>)

Word: python
Token ID: 3
Embedding:
tensor([-0.8552, -1.0608,  0.0716,  1.0493,  1.4282, -0.8662,  0.9909,  0.3830],
       grad_fn=<UnbindBackward0>)


In [52]:
import torch
import torch.nn as nn

embedding = nn.Embedding(
    num_embeddings=12,
    embedding_dim=8
)

tokens = torch.tensor([1, 5, 3])

output = embedding(tokens)

print("Embedding matrix:")
print(embedding.weight)

print("\nSelected rows:")
print(output)

print("\nRow 1:")
print(embedding.weight[1])

print("\nRow 5:")
print(embedding.weight[5])

print("\nRow 3:")
print(embedding.weight[3])

Embedding matrix:
Parameter containing:
tensor([[ 0.5614, -0.4193, -0.6228,  1.3272,  0.1458, -0.1898, -0.6729, -0.1422],
        [-0.4564,  1.0058,  0.3596, -0.1355,  0.4440, -2.5077, -0.2619,  1.0314],
        [ 1.5880,  2.0084, -0.2696, -0.3015, -2.2540, -0.0236,  0.6592,  0.8988],
        [ 1.4962, -0.8751, -0.1186, -1.4491, -0.5258, -0.7114, -0.7783,  0.8523],
        [ 0.2410, -0.2244,  1.9672,  0.5847, -0.6116,  0.5552,  1.1814, -1.0251],
        [-0.2131, -0.0815,  0.6400, -0.1734, -0.8617,  0.5422,  0.8897,  0.1000],
        [-0.4676, -1.3438, -0.6831,  0.9233, -0.7179, -0.8954,  0.6829,  0.3028],
        [ 1.8040,  0.9145, -0.2456,  0.4959,  0.1048, -0.7927, -1.0560, -0.9324],
        [ 0.6940, -0.5018, -0.5362, -0.5495, -1.3226, -0.7196,  1.6014,  2.1774],
        [ 1.3388,  1.7372,  0.6586, -1.2091,  0.0705,  0.2092,  0.7246, -0.1081],
        [-0.2035, -0.3576,  0.7105,  1.5056,  0.4687, -1.7245, -0.4941, -0.0623],
        [-0.2364,  0.4572,  0.3487,  1.0479, -2.0192, -2.5

#  Program 6 — RNN Encoder

In [1]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
  def __init__(self,vocab_size,embedding_dim,hidden_size):
     super().__init__()
     self.embedding=nn.Embedding(vocab_size,embedding_dim)
     self.rnn=nn.GRU(embedding_dim,hidden_size,batch_first=True)

  def forward(self,x):
    embedded=self.embedding(x)
    output,hidden=self.rnn(embedded)
    return output,hidden

In [3]:
encoder=Encoder(vocab_size=100,embedding_dim=128,hidden_size=256)
x = torch.tensor([
    [5, 10, 20, 15]
])

outputs, hidden = encoder(x)

print("Outputs:", outputs.shape)
print("Hidden:", hidden.shape)

Outputs: torch.Size([1, 4, 256])
Hidden: torch.Size([1, 1, 256])


# Program 7 — RNN Decoder

In [5]:
class Decoder(nn.Module):

    def __init__(self,vocab_size,embedding_dim,hidden_dim):
       super().__init__()
       self.embedding = nn.Embedding(vocab_size,embedding_dim)
       self.rnn=nn.RNN(embedding_dim,hidden_dim,batch_first=True)
       self.fc=nn.Linear(hidden_dim,vocab_size)

    def forward(self, x, hidden):

        embedded = self.embedding(x)

        output, hidden = self.rnn(
            embedded,
            hidden
        )

        prediction = self.fc(output)

        return prediction, hidden

# Program 8 — LSTM Encoder

In [6]:
class LSTMEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

    def forward(self, x):

        embedded = self.embedding(x)

        outputs, (hidden, cell) = \
            self.lstm(embedded)

        return outputs, hidden, cell